# Principal Component Analysis

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/pca-dimensionality/01-pca

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'

## Intuition — find the directions that matter

High-dimensional data often really lives on a lower-dimensional surface — many features are
correlated and redundant. **PCA** finds the orthogonal directions (**principal components**) along
which the data varies most, ranks them by variance, and lets you keep just the top few. Mechanically
it's the **eigendecomposition of the covariance matrix** (you built this in the linear-algebra course):
the top eigenvector is the direction of maximum variance, its eigenvalue is *how much* variance, and
projecting onto the top `m` components compresses the data while preserving most of its spread. We
derive PC1 by hand, validate against `sklearn`, and see how far it compresses real data.

## PCA via Eigendecomposition

1. Center the data
2. Compute covariance matrix
3. Eigendecompose
4. Project onto top-k eigenvectors

In [ ]:
np.random.seed(42)
n = 200
t = np.linspace(0, 2 * np.pi, n)
X = np.column_stack([3 * np.cos(t), 1.5 * np.sin(t)]) + np.random.randn(n, 2) * 0.3

X_centered = X - X.mean(axis=0)
cov = np.cov(X_centered.T)
eigenvalues, eigenvectors = np.linalg.eigh(cov)
idx = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

X_pca = X_centered @ eigenvectors[:, :2]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# Original data with principal components
ax = axes[0]
ax.scatter(X[:, 0], X[:, 1], c='#818cf8', s=10, alpha=0.5)
mean = X.mean(axis=0)
for i in range(2):
    v = eigenvectors[:, i] * np.sqrt(eigenvalues[i]) * 2
    ax.arrow(mean[0], mean[1], v[0], v[1], head_width=0.2, head_length=0.1,
             fc='#f43f5e' if i == 0 else '#14b8a6', ec='white', linewidth=1.5)
ax.set_title('Original + PCs', color='white', fontsize=11)
ax.set_aspect('equal')

# Projected onto PC1
ax = axes[1]
ax.scatter(X_pca[:, 0], np.zeros_like(X_pca[:, 0]), c='#818cf8', s=10, alpha=0.5)
ax.set_title('Projected onto PC1', color='white', fontsize=11)
ax.set_xlabel('$z_1$')

# Explained variance
ax = axes[2]
var_ratio = eigenvalues / eigenvalues.sum()
ax.bar(range(1, len(var_ratio) + 1), var_ratio, color=['#f43f5e', '#14b8a6', '#eab308'][:len(var_ratio)])
ax.plot(range(1, len(var_ratio) + 1), np.cumsum(var_ratio), 'o-', color='white', linewidth=1.5)
ax.set_xlabel('Component')
ax.set_ylabel('Variance Explained')
ax.set_title('Scree Plot', color='white', fontsize=11)
ax.axhline(0.95, color='#94a3b8', linestyle='--', alpha=0.5, label='95% threshold')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()
print(f'PC1 explains {var_ratio[0]*100:.1f}% of variance')
print(f'PC1+PC2 explain {sum(var_ratio[:2])*100:.1f}% of variance')

**What to notice:** **PC1** (red arrow) points along the ellipse's **long axis** — the direction of
greatest variance — and **PC2** (teal) is the short perpendicular axis. Projecting onto PC1 alone (middle
panel) keeps almost all the spread, collapsing 2-D to 1-D with little loss. The bar chart shows PC1
captures the large majority of the variance.

## The library way — validate against `sklearn`

`sklearn.decomposition.PCA` computes the same thing (via SVD). The cell checks our
eigendecomposition's explained-variance ratios and component directions match it (components are unique
only up to sign).

In [ ]:
from sklearn.decomposition import PCA

sk = PCA(n_components=2).fit(X)
our_ratio = eigenvalues / eigenvalues.sum()
print('our explained-variance ratio    :', our_ratio.round(4))
print('sklearn explained-variance ratio:', sk.explained_variance_ratio_.round(4))
assert np.allclose(our_ratio, sk.explained_variance_ratio_, atol=1e-6), "variance ratios must match"

for i in range(2):   # components match up to an arbitrary sign
    aligned = np.allclose(eigenvectors[:, i], sk.components_[i]) or \
              np.allclose(eigenvectors[:, i], -sk.components_[i])
    assert aligned, f"component {i} must match sklearn (up to sign)"
print('\nour eigendecomposition PCA == sklearn PCA (variance + directions) ✓')

**What to notice:** identical explained-variance ratios and matching component directions (up to
sign) — PCA via covariance eigendecomposition *is* what `sklearn` does (it uses SVD, the numerically
preferred route from the SVD lesson, but the answer is the same).

## Why PC1 is the top eigenvector — by hand

The variance along a unit direction $v$ is $\mathrm{Var}(\tilde X v) = v^\top C v$. Maximizing it under $\lVert v\rVert=1$ with a Lagrange multiplier gives $Cv=\lambda v$ — so $v$ is an eigenvector of the covariance and the captured variance equals $\lambda$. Pick the **largest** $\lambda$ for PC1.

Below we take the lesson's 4-point example, get the eigenvalues straight from the characteristic polynomial $\lambda^2-(\mathrm{tr}\,C)\lambda+\det C=0$, and confirm the projected variance equals the eigenvalue and that `sklearn` agrees.

In [ ]:
import numpy as np
from sklearn.decomposition import PCA

Xe = np.array([[2, 1], [1, 2], [-1, -2], [-2, -1]], dtype=float)  # already centered
n = len(Xe)
C = (Xe.T @ Xe) / (n - 1)                 # = np.cov(Xe.T)
print('covariance C =\n', np.round(C, 3))

# Eigenvalues by hand: lambda^2 - tr*lambda + det = 0  (2x2 quadratic formula)
tr, det = np.trace(C), np.linalg.det(C)
disc = np.sqrt(tr**2 - 4 * det)
lam = sorted([(tr + disc) / 2, (tr - disc) / 2], reverse=True)
print(f'tr={tr:.3f}, det={det:.3f}  ->  lambda1={lam[0]:.3f}, lambda2={lam[1]:.3f}')
print('eigvals (np.linalg.eigh):', np.round(np.linalg.eigh(C)[0][::-1], 3))

# Eigenvector for lambda1 from (C - lambda*I) v = 0, then verify projected variance == lambda
w, V = np.linalg.eigh(C)
v1 = V[:, -1]                              # eigenvector of largest eigenvalue
print('\nPC1 direction (normalized):', np.round(v1 / np.abs(v1).max(), 3), ' (proportional to (1,1))')
proj_var = np.var(Xe @ v1, ddof=1)
print(f'variance of projection onto PC1 = {proj_var:.3f}  ==  lambda1 = {lam[0]:.3f}')
print(f'variance ratio kept by PC1 = {lam[0] / (lam[0] + lam[1]):.3f}  (90%)')

# sklearn cross-check
pca = PCA(n_components=2).fit(Xe)
print('\nsklearn explained_variance_:', np.round(pca.explained_variance_, 3))
print('sklearn variance ratio     :', np.round(pca.explained_variance_ratio_, 3))


**What to notice:** the by-hand derivation shows *why* PC1 is the **top eigenvector** — it's the
unit direction maximizing the projected variance `wᵀ Σ w`, and the Lagrangian for that constraint is
solved exactly by the leading eigenvector of the covariance `Σ`. The eigenvalue *is* the variance along
that direction.

## Explained variance and the scree plot

Each principal component captures a share of the total variance. The cumulative curve tells you how many components to keep.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.datasets import load_digits

X = load_digits().data            # 64 features
pca = PCA().fit(X)
cum = np.cumsum(pca.explained_variance_ratio_)
k95 = np.argmax(cum >= 0.95) + 1
print(f'{k95} components explain 95% of variance (of {X.shape[1]})')

plt.plot(range(1, len(cum)+1), cum, color='#818cf8')
plt.axhline(0.95, ls='--', color='#f43f5e'); plt.axvline(k95, ls='--', color='#14b8a6')
plt.xlabel('components'); plt.ylabel('cumulative explained variance')
plt.title('Scree / cumulative variance'); plt.show()

**What to notice:** on the 64-pixel digits data, only ~**29 components explain 95%** of the
variance — better than 2× compression with almost no information lost. That's PCA's practical payoff:
real high-dimensional data has far fewer *effective* dimensions than raw features.

## Gotchas & tradeoffs

- **Standardize first (usually).** PCA maximizes variance, so a feature on a larger scale dominates the
  components. Standardize unless the features are already in comparable units.
- **PCA is linear.** It finds flat subspaces; data on a curved manifold (a swiss roll) needs non-linear
  methods (t-SNE/UMAP, next lesson).
- **Components are combinations of all features** — powerful for compression, but harder to interpret
  than the original features.
- **Variance ≠ importance.** PCA assumes high-variance directions carry the signal; sometimes the
  discriminative direction has *low* variance (then use LDA/supervised methods).

In [ ]:
# PCA is scale-sensitive: an un-standardized large-scale feature hijacks PC1
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
np.random.seed(0)
A = np.random.randn(300, 2)
A[:, 1] *= 100                                   # feature 2 on a 100x larger scale
raw = PCA(2).fit(A).explained_variance_ratio_
std = PCA(2).fit(StandardScaler().fit_transform(A)).explained_variance_ratio_
print(f'raw          : PC1 explains {raw[0]:.3f} of variance (all from the large-scale feature)')
print(f'standardized : PC1 explains {std[0]:.3f} (features now on equal footing)')

**What to notice:** without standardization PC1 captures ~100% of the "variance" — but only because
one feature was on a 100× scale, not because it's more informative. After standardizing, the two
features contribute comparably. Scaling is not optional for PCA unless your features are already
commensurable.

## Key takeaways

- PCA finds orthogonal directions (**eigenvectors of the covariance**) of maximum variance.
- Projecting onto the top $k$ components reduces dimensions while keeping most variance.
- Choose $k$ from the **cumulative explained variance** (e.g. 95%) or the scree elbow.
- **Standardize first**; PCA is linear and unsupervised, computed efficiently via **SVD**.

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — The covariance matrix

PCA starts from the covariance of **centered** data:

$$C = \frac{1}{n} X_c^\top X_c, \qquad X_c = X - \bar{X}$$

Implement it. The checks verify it against `np.cov`, its symmetry, and that the diagonal holds the per-feature variances.

In [ ]:
def covariance(X):
    """Covariance matrix (population, 1/n) of the rows of X."""
    X = np.asarray(X, dtype=float)

    # TODO(you): subtract the column means
    Xc = ...

    # TODO(you): Xc^T Xc / n
    return ...

In [ ]:
# Checks — run me
rng = np.random.default_rng(0)
Xp = rng.multivariate_normal([1, -2], [[3, 1], [1, 2]], size=500)

C = covariance(Xp)
assert C.shape == (2, 2) and np.allclose(C, C.T), "covariance is square and symmetric"
assert np.allclose(C, np.cov(Xp.T, ddof=0)), "must match np.cov with ddof=0"
assert np.allclose(np.diag(C), Xp.var(axis=0)), "the diagonal holds the per-feature variances"
# Edge case: a constant (zero-variance) feature contributes zero variance and
# zero covariance with everything else
X_zero_var = np.column_stack([np.array([1.0, 2.0, 3.0, 4.0]), np.full(4, 5.0)])
C_zero_var = covariance(X_zero_var)
assert np.allclose(C_zero_var[1, :], 0) and np.allclose(C_zero_var[:, 1], 0), \
    "a constant feature has zero variance and zero covariance with anything"

# Edge case: perfectly correlated features make the covariance matrix singular
# (rank-deficient) -- there's only one real direction of variation
X_corr = np.column_stack([np.array([1.0, 2.0, 3.0, 4.0]), np.array([2.0, 4.0, 6.0, 8.0])])
C_corr = covariance(X_corr)
assert abs(np.linalg.det(C_corr)) < 1e-10, "perfectly correlated features give a singular covariance matrix"
assert np.linalg.matrix_rank(C_corr) == 1, "only one direction carries any variance"

print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def covariance(X):
    X = np.asarray(X, dtype=float)
    Xc = X - X.mean(axis=0)
    return Xc.T @ Xc / len(X)
```

</details>

### Exercise 2 — Explained variance and choosing m

Each eigenvalue is the variance captured along its component, so the **explained variance ratio** is the sorted eigenvalues divided by their sum — and "how many components for 90%?" is just the first index where the cumulative sum crosses the threshold. Implement both halves of the scree-plot workflow.

In [ ]:
def explained_variance_ratio(eigvals):
    """Eigenvalues sorted descending, normalized to sum to 1."""
    lams = np.sort(np.asarray(eigvals, dtype=float))[::-1]

    # TODO(you): normalize
    return ...


def components_for(eigvals, threshold):
    """Smallest m whose top-m components explain >= threshold of the variance."""
    evr = explained_variance_ratio(eigvals)

    # TODO(you): first index where the cumulative sum reaches threshold, plus 1
    # (hint: np.cumsum + np.searchsorted)
    return ...

In [ ]:
# Checks — run me
assert np.allclose(explained_variance_ratio([1.0, 3.0]), [0.75, 0.25]), "sorted descending, normalized"
assert abs(sum(explained_variance_ratio([5, 2, 1, 0.5])) - 1) < 1e-12, "ratios sum to 1"
assert components_for([5.0, 3.0, 1.0, 1.0], 0.8) == 2, "5+3 of 10 covers 80% at m=2"
assert components_for([5.0, 3.0, 1.0, 1.0], 0.95) == 4, "the tail matters for 95%"
# Edge case: tied eigenvalues -- every component contributes equally
assert components_for([2.0, 2.0, 2.0, 2.0], 0.5) == 2, "with ties, half the components explain half the variance"

# Edge case: a single component (nothing left to reduce)
assert components_for([5.0], 0.999) == 1, "one component always explains 100% of the variance"

# Edge case: zero-variance directions (rank-deficient input) never need to be kept
assert components_for([5.0, 0.0, 0.0], 0.999) == 1, "zero eigenvalues add nothing to keep"

print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def explained_variance_ratio(eigvals):
    lams = np.sort(np.asarray(eigvals, dtype=float))[::-1]
    return lams / lams.sum()


def components_for(eigvals, threshold):
    evr = explained_variance_ratio(eigvals)
    return int(np.searchsorted(np.cumsum(evr), threshold) + 1)
```

</details>

---
## 🌐 Extra practice — from Open-Deep-ML

A [DML](https://github.com/Open-Deep-ML/DML-OpenProblem)-style drill that complements the eigendecomposition work above: PCA packaged into a single function matching DML's exact signature and return convention.

### Exercise 3 — PCA as a single function (DML #19)

DML's [problem 19](https://github.com/Open-Deep-ML/DML-OpenProblem/tree/main/questions/19_principal-component-analysis-pca-implementation) packages PCA into one function `pca(data, k)` with two differences from the `covariance()` helper above:

- it **standardizes** each column to zero mean / unit variance ($z$-score) before building the covariance matrix, instead of only centering
- it returns the **eigenvectors themselves** — the top-`k` principal-component directions, sorted by decreasing eigenvalue and rounded to 4 decimal places — not the projected data

`np.linalg.eig`'s eigenvectors are only defined up to a sign flip, so match DML's convention by sorting and slicing without flipping any signs yourself.

In [ ]:
def dml_pca(data, k):
    """DML #19 signature: z-score standardize, return the top-k eigenvectors
    of the covariance matrix (principal-component directions), sorted by
    decreasing eigenvalue, rounded to 4 decimal places."""
    data = np.asarray(data, dtype=float)

    # TODO(you): standardize each column: (x - mean) / std
    data_standardized = ...

    # TODO(you): covariance matrix of the standardized data (rowvar=False -> features as columns)
    cov = ...

    # TODO(you): eigendecompose, then sort both eigenvalues and eigenvectors by decreasing eigenvalue
    eigenvalues, eigenvectors = np.linalg.eig(cov)
    ...

    # TODO(you): keep the first k columns (principal components) and round to 4 decimals
    principal_components = ...
    return principal_components

In [ ]:
# Checks — run me (matches DML's tests.json exactly)
out1 = dml_pca(np.array([[1, 2], [3, 4], [5, 6]]), 1)
assert np.allclose(out1, [[0.7071], [0.7071]]), "DML tests.json case 1"

out2 = dml_pca(np.array([[4, 2, 1], [5, 6, 7], [9, 12, 1], [4, 6, 7]]), 2)
assert np.allclose(out2, [[0.6855, 0.0776], [0.6202, 0.4586], [-0.3814, 0.8853]]), "DML tests.json case 2"

# Sanity checks on the returned directions: unit-length and mutually orthogonal,
# as eigenvectors of a symmetric (covariance) matrix must be
assert np.allclose(np.linalg.norm(out2, axis=0), 1, atol=1e-3), "each principal component is unit-length"
assert abs(out2[:, 0] @ out2[:, 1]) < 1e-3, "top-2 components are orthogonal"

# Variance-explained ratio check: reuse Exercise 2's explained_variance_ratio on
# this dataset's standardized-covariance eigenvalues
data3 = np.array([[4, 2, 1], [5, 6, 7], [9, 12, 1], [4, 6, 7]], dtype=float)
Xs = (data3 - data3.mean(axis=0)) / data3.std(axis=0)
eigvals_all = np.linalg.eigvalsh(np.cov(Xs, rowvar=False))[::-1]
evr = explained_variance_ratio(eigvals_all)
assert evr[0] > evr[1] > 0, "the first component explains more variance than the second"
assert abs(evr.sum() - 1) < 1e-9, "explained-variance ratios still sum to 1"

# Edge case: requesting more components than there are features. DML's slice
# `eigenvectors_sorted[:, :k]` just saturates at the number of columns available --
# no error, no padding, min(k, n_features) components come back.
out_over = dml_pca(data3, k=10)
assert out_over.shape == (3, 3), "k > n_features silently returns all n_features components"

print("✅ Exercise 3 passed")

<details>
<summary>💡 Show solution</summary>

```python
def dml_pca(data, k):
    data = np.asarray(data, dtype=float)
    data_standardized = (data - data.mean(axis=0)) / data.std(axis=0)
    cov = np.cov(data_standardized, rowvar=False)
    eigenvalues, eigenvectors = np.linalg.eig(cov)
    idx = np.argsort(eigenvalues)[::-1]
    eigenvectors_sorted = eigenvectors[:, idx]
    principal_components = eigenvectors_sorted[:, :k]
    return np.round(principal_components, 4)
```

</details>